# HerittageFNS — Bước 3: Suy Luận Thực Tế & Giải Thích Bằng Hệ Luật Mờ Trên 18 Mẫu Hold-out

Notebook này thực hiện kiểm nghiệm thực tế và giải thích suy luận (Explainable AI - XAI) trên **18 mẫu bài hát độc lập** (3 mẫu/thể loại) đã được tách riêng ở Bước 1:
1. **Trích xuất 167 đặc trưng âm học**: Trích xuất phân đoạn 12 giây tiêu biểu trên từng bài hát hold-out.
2. **Chiếu không gian mờ (FCM Projection)**:
   - Dạng **Crisp (7 trục nhãn chữ)**: Nhãn ngữ nghĩa ngôn ngữ tự nhiên (Energy, Harmony, Pitch, Spectral, Tempo, Timbre, Vibrato).
   - Dạng **Continuous (27 mức độ mờ)**: Giá trị độ thuộc liên tục trong khoảng [0, 1].
3. **Đối chiếu & Kích hoạt hệ luật mờ**:
   - Đối chiếu với hệ luật tổng hợp từ **Random Forest Crisp (1.408 luật)**.
   - Đối chiếu với hệ luật chất lượng cao từ **Random Forest Continuous (718 luật)**.
4. **Xếp hạng & Thống kê Top 5 luật phù hợp nhất cho từng mẫu**:
   - Ưu tiên luật dự đoán **ĐÚNG** thể loại thực tế (True Positive).
   - Cảnh báo rõ ràng các luật kích hoạt dự đoán **KHÁC** thể loại gốc (False Positive / Nhầm lẫn).
5. **Xuất báo cáo**: Lưu bảng tổng hợp `holdout_top5_rules_summary.csv` và JSON chi tiết vào thư mục `data/rules/`.


In [ ]:
import os, sys, json, ast, warnings, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt

from sklearn.preprocessing import MinMaxScaler
from skfuzzy import cmeans_predict
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Thiết lập theme tối nhất quán
plt.style.use('dark_background')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Segoe UI']
plt.rcParams['axes.edgecolor']  = '#30363d'
plt.rcParams['axes.facecolor']  = '#0d1117'
plt.rcParams['figure.facecolor']= '#0d1117'
plt.rcParams['text.color']      = '#e6edf3'
plt.rcParams['axes.labelcolor'] = '#e6edf3'
plt.rcParams['xtick.color']     = '#8b949e'
plt.rcParams['ytick.color']     = '#8b949e'

# Đảm bảo các thư mục tồn tại
for d in ['./data/datasets', './data/models', './data/plots', './data/rules']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Môi trường và thư mục sẵn sàng!')


## 1. Nạp 18 Mẫu Hold-out & Trích Xuất 167 Đặc Trưng Âm Học


In [ ]:
# Nạp danh sách 18 mẫu hold-out độc lập
df_holdout_meta = pd.read_csv('./data/datasets/holdout_samples_for_inference.csv')
print(f'Nạp thành công {len(df_holdout_meta)} mẫu kiểm thử độc lập (3 mẫu × 6 thể loại):')
print(df_holdout_meta[['label', 'audio']].to_string())

# Hàm trích xuất 167 đặc trưng an toàn cho 1 file audio
def extract_audio_features(loc_path):
    try:
        info = sf.info(loc_path)
        dur = float(info.duration)
        sr_actual = int(info.samplerate)
    except Exception:
        dur = 45.0
        sr_actual = 16000

    # Lấy phân đoạn 12s tiêu biểu (5.0s - 17.0s)
    s_time = 5.0 if dur >= 18.0 else 0.0
    e_time = min(dur, s_time + 12.0)
    start_frame = int(s_time * sr_actual)
    stop_frame  = int(e_time * sr_actual)

    data, sr = sf.read(loc_path, start=start_frame, stop=stop_frame, dtype='float32')
    if data.ndim > 1:
        data = np.mean(data, axis=1)

    if len(data) == 0 or len(data) < int(sr * 1.0):
        data, sr = sf.read(loc_path, dtype='float32')
        if data.ndim > 1: data = np.mean(data, axis=1)

    min_len = int(sr * 3.0)
    if len(data) < min_len and len(data) > 0:
        repeats = int(np.ceil(min_len / len(data)))
        data = np.tile(data, repeats)[:int(sr * 12.0)]

    arr, _ = librosa.effects.trim(data, top_db=20)
    if len(arr) == 0: arr = data

    if len(arr) > 0:
        mx = float(np.max(np.abs(arr)))
        if mx > 1e-6: arr = arr / mx
    else:
        arr = np.zeros(int(sr * 12.0), dtype=np.float32)

    feats = {}
    f0 = None
    try:
        f0, voiced, _ = librosa.pyin(arr, sr=sr, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'),
                                     frame_length=1024, hop_length=512)
        f0v = f0[voiced] if voiced.any() else np.array([])
        feats['pitch_mean']   = float(np.nanmean(f0v)) if len(f0v) else 0.0
        feats['pitch_std']    = float(np.nanstd(f0v))  if len(f0v) else 0.0
        feats['pitch_range']  = float(np.nanmax(f0v) - np.nanmin(f0v)) if len(f0v) else 0.0
        feats['pitch_median'] = float(np.nanmedian(f0v)) if len(f0v) else 0.0
        feats['voiced_ratio'] = float(voiced.mean()) if len(voiced) else 0.0
    except Exception:
        feats.update({'pitch_mean':0.0,'pitch_std':0.0,'pitch_range':0.0,'pitch_median':0.0,'voiced_ratio':0.0})

    try:
        tempo, beats = librosa.beat.beat_track(y=arr, sr=sr)
        bt = librosa.frames_to_time(beats, sr=sr)
        feats['tempo']           = float(np.squeeze(tempo))
        feats['beat_count']      = len(beats)
        ioi = np.diff(bt)
        feats['beat_regularity'] = float(1.0 - np.std(ioi)/(np.mean(ioi)+1e-6)) if len(ioi)>1 else 0.0
    except Exception:
        feats.update({'tempo':0.0,'beat_count':0,'beat_regularity':0.0})

    try:
        if f0 is not None and len(f0) > 30:
            fps = sr / 512
            vf0 = np.where(np.isnan(f0), 0, f0)
            idx_v = np.where(vf0 > 0)[0]
            if len(idx_v) > 10:
                f0i = np.interp(np.arange(len(vf0)), idx_v, vf0[idx_v])
                lo, hi = 4/(fps/2), min(10/(fps/2), 0.99)
                if lo < hi:
                    b, a = butter(2, [lo, hi], btype='band')
                    vib = filtfilt(b, a, f0i)
                    feats['vibrato_extent'] = float(np.std(vib))
                    feats['vibrato_energy'] = float(np.mean(vib**2))
                else: feats.update({'vibrato_extent':0.0,'vibrato_energy':0.0})
            else: feats.update({'vibrato_extent':0.0,'vibrato_energy':0.0})
        else: feats.update({'vibrato_extent':0.0,'vibrato_energy':0.0})
    except Exception:
        feats.update({'vibrato_extent':0.0,'vibrato_energy':0.0})

    try:
        rms = librosa.feature.rms(y=arr)[0]
        feats['energy_mean']   = float(np.mean(rms))
        feats['energy_std']    = float(np.std(rms))
        feats['dynamic_range'] = float(np.ptp(rms))
        feats['energy_skew']   = float(pd.Series(rms).skew())
    except Exception:
        feats.update({'energy_mean':0.0,'energy_std':0.0,'dynamic_range':0.0,'energy_skew':0.0})

    try:
        mfcc   = librosa.feature.mfcc(y=arr, sr=sr, n_mfcc=40)
        dmfcc  = librosa.feature.delta(mfcc)
        d2mfcc = librosa.feature.delta(mfcc, order=2)
        for i in range(40):
            feats[f'mfcc_{i+1:02d}_mean'] = float(np.mean(mfcc[i]))
            feats[f'mfcc_{i+1:02d}_std']  = float(np.std(mfcc[i]))
        for i in range(13):
            feats[f'dmfcc_{i+1:02d}_mean']  = float(np.mean(dmfcc[i]))
            feats[f'd2mfcc_{i+1:02d}_mean'] = float(np.mean(d2mfcc[i]))
    except Exception: pass

    try:
        chroma = librosa.feature.chroma_stft(y=arr, sr=sr)
        NOTES  = ['C','Cs','D','Ds','E','F','Fs','G','Gs','A','As','B']
        for i, n in enumerate(NOTES):
            feats[f'chroma_{n}_mean'] = float(np.mean(chroma[i]))
            feats[f'chroma_{n}_std']  = float(np.std(chroma[i]))
        feats['chroma_std_overall'] = float(np.std(chroma))
    except Exception: pass

    try:
        sc  = librosa.feature.spectral_centroid(y=arr, sr=sr)[0]
        sb  = librosa.feature.spectral_bandwidth(y=arr, sr=sr)[0]
        sro = librosa.feature.spectral_rolloff(y=arr, sr=sr)[0]
        sct = librosa.feature.spectral_contrast(y=arr, sr=sr)
        sf_ = librosa.feature.spectral_flatness(y=arr)[0]
        feats['spectral_centroid_mean']  = float(np.mean(sc))
        feats['spectral_centroid_std']   = float(np.std(sc))
        feats['spectral_bandwidth_mean'] = float(np.mean(sb))
        feats['spectral_rolloff_mean']   = float(np.mean(sro))
        feats['spectral_flatness_mean']  = float(np.mean(sf_))
        for i in range(sct.shape[0]):
            feats[f'spectral_contrast_{i+1}'] = float(np.mean(sct[i]))
    except Exception: pass

    try:
        zcr = librosa.feature.zero_crossing_rate(arr)[0]
        feats['zcr_mean'] = float(np.mean(zcr))
        feats['zcr_std']  = float(np.std(zcr))
    except Exception: pass

    try:
        tz = librosa.feature.tonnetz(y=arr, sr=sr)
        for i in range(6):
            feats[f'tonnetz_{i+1}'] = float(np.mean(tz[i]))
    except Exception: pass

    try:
        mel    = librosa.feature.melspectrogram(y=arr, sr=sr, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        feats['mel_mean'] = float(np.mean(mel_db))
        feats['mel_std']  = float(np.std(mel_db))
    except Exception: pass

    return feats

print('\nBắt đầu trích xuất 167 đặc trưng cho 18 mẫu kiểm thử...')
holdout_rows = []
for idx, r in tqdm(df_holdout_meta.iterrows(), total=len(df_holdout_meta), desc='Extracting Holdout'):
    audio_path = r['audio']
    hf_path = audio_path if audio_path.startswith('audios/') else f'audios/{audio_path}'
    local_path = hf_hub_download(repo_id='NTQAI/Vietnamese-Traditional-Music', filename=hf_path, repo_type='dataset')
    
    feats = extract_audio_features(local_path)
    feats['sample_idx'] = idx
    feats['label']      = r['label']
    feats['audio_path'] = audio_path
    feats['text']       = r.get('text', '')
    holdout_rows.append(feats)

df_holdout_feats = pd.DataFrame(holdout_rows).fillna(0)
df_holdout_feats.to_csv('./data/datasets/holdout_features_all.csv', index=False)
print(f'-> Đã lưu 167 đặc trưng của 18 mẫu: ./data/datasets/holdout_features_all.csv ({df_holdout_feats.shape[0]} mẫu × {df_holdout_feats.shape[1]} cột)')


## 2. Chiếu Không Gian Mờ: Tạo Biểu Diễn Crisp (7 Trục) & Continuous (27 Mức Độ Mờ)


In [ ]:
# 1. Nạp mô hình FCM và danh sách đặc trưng từ Bước 1
with open('./data/models/fcm_results.pkl', 'rb') as f:
    fcm_res = pickle.load(f)

# Nạp dữ liệu huấn luyện để khớp MinMaxScaler chuẩn xác
df_train_feats = pd.read_csv('./data/datasets/features_all.csv')
feat_cols = [c for c in df_train_feats.columns if c not in ['label', 'sample_idx', 'audio_path', 'segment_id', 'segment_time'] and not c.startswith('_') and pd.api.types.is_float_dtype(df_train_feats[c])]

scaler = MinMaxScaler()
scaler.fit(df_train_feats[feat_cols].fillna(0).values)

feat_col_idx = {c: i for i, c in enumerate(feat_cols)}

# Chuẩn hóa đặc trưng của 18 mẫu hold-out theo đúng scaler huấn luyện
X_holdout_scaled = scaler.transform(df_holdout_feats[feat_cols].fillna(0).values)

# Cấu hình 7 trục và tên mức độ ngôn ngữ
FUZZY_AXES = {
    'Energy':   ['energy_mean', 'dynamic_range'],
    'Harmony':  [f'chroma_{n}_mean' for n in ['C','D','E','F','G','A','B']],
    'Pitch':    ['pitch_mean', 'pitch_range'],
    'Spectral': ['spectral_centroid_mean', 'spectral_bandwidth_mean', 'spectral_rolloff_mean'],
    'Tempo':    ['tempo'],
    'Timbre':   [f'mfcc_{i:02d}_mean' for i in range(1, 14)],
    'Vibrato':  ['vibrato_extent', 'vibrato_energy'],
}

MUSIC_L = {
    'Energy':   {5: ['Rất yếu', 'Yếu', 'Vừa', 'Mạnh', 'Rất mạnh']},
    'Harmony':  {3: ['Thấp', 'Trung bình', 'Cao']},
    'Pitch':    {5: ['Rất trầm', 'Trầm', 'Trung', 'Cao', 'Rất cao']},
    'Spectral': {3: ['Thấp', 'Trung bình', 'Cao']},
    'Tempo':    {3: ['Chậm', 'Vừa', 'Nhanh']},
    'Timbre':   {5: ['Rất thấp', 'Thấp', 'Trung bình', 'Cao', 'Rất cao']},
    'Vibrato':  {3: ['Ít luyến', 'Vừa', 'Nhiều luyến']},
}

# 2. Chiếu từng mẫu lên các tâm cụm FCM để tính hàm liên thuộc (Membership)
crisp_dict = {
    'sample_idx': df_holdout_feats['sample_idx'],
    'label': df_holdout_feats['label'],
    'audio_path': df_holdout_feats['audio_path']
}
continuous_dict = {
    'sample_idx': df_holdout_feats['sample_idx'],
    'label': df_holdout_feats['label'],
    'audio_path': df_holdout_feats['audio_path']
}

print('Chiếu 18 mẫu vào 7 không gian mờ...')
for ax_name, fcm in fcm_res.items():
    k = fcm['k']
    cntr = fcm['centers'] # (k, n_feats)
    vf = [f for f in FUZZY_AXES[ax_name] if f in feat_col_idx]
    cidx = [feat_col_idx[f] for f in vf]
    X_ax = np.nan_to_num(X_holdout_scaled[:, cidx]).T # (n_feats, 18)
    
    # Dự đoán độ thuộc u bằng cmeans_predict
    u, _, _, _, _, _ = cmeans_predict(X_ax, cntr_trained=cntr, m=2.0, error=0.005, maxiter=1000, seed=42) # (k, 18)
    
    cnames = fcm['names']
    # 1. Nhãn Crisp (argmax)
    hard_idx = np.argmax(u, axis=0)
    crisp_dict[ax_name] = [cnames[idx] for idx in hard_idx]
    
    # 2. Mức độ mờ liên tục [0, 1]
    names = MUSIC_L[ax_name][k]
    for lvl_name in names:
        c_orig = next(idx for idx, name in cnames.items() if name == lvl_name)
        col_name = f"{ax_name}_{lvl_name}".replace(' ', '_')
        continuous_dict[col_name] = np.round(u[c_orig, :], 4)

df_holdout_crisp = pd.DataFrame(crisp_dict)
df_holdout_continuous = pd.DataFrame(continuous_dict)

# Lưu 2 bảng mờ của tập hold-out
df_holdout_crisp.to_csv('./data/datasets/holdout_fuzzy_crisp.csv', index=False)
df_holdout_continuous.to_csv('./data/datasets/holdout_fuzzy_continuous.csv', index=False)

print(f'\n-> Đã tạo ./data/datasets/holdout_fuzzy_crisp.csv: {df_holdout_crisp.shape[0]} mẫu × {df_holdout_crisp.shape[1]} cột')
print(f'-> Đã tạo ./data/datasets/holdout_fuzzy_continuous.csv: {df_holdout_continuous.shape[0]} mẫu × {df_holdout_continuous.shape[1]} cột')

print('\nBẢNG ĐẶC TRƯNG NGỮ NGHĨA RỜI RẠC (CRISP) CỦA 18 MẪU:')
display_cols = ['sample_idx', 'label', 'Tempo', 'Pitch', 'Energy', 'Vibrato', 'Timbre', 'Harmony', 'Spectral']
print(df_holdout_crisp[display_cols].to_string(index=False))


## 3. Đối Chiếu Hệ Luật & Xếp Hạng Top 5 Luật Phù Hợp Nhất Cho Từng Mẫu


In [ ]:
# Nạp 2 bộ luật đã tinh lọc từ Random Forest
with open('./data/rules/rules_random_forest_crisp_aggregated.json', 'r', encoding='utf-8') as f:
    rf_crisp_rules = json.load(f)

with open('./data/rules/rules_random_forest_aggregated.json', 'r', encoding='utf-8') as f:
    rf_cont_rules = json.load(f)

print(f'Đã nạp {len(rf_crisp_rules)} luật Crisp và {len(rf_cont_rules)} luật Continuous.')

# 1. Hàm kiểm tra luật Crisp
def match_crisp_rule(rule_text, s_crisp):
    if rule_text == 'TRUE': return True
    parts = rule_text.split(' AND ')
    for part in parts:
        if ' == ' in part:
            feat, val = part.split(' == ')
            feat = feat.strip()
            val = val.strip().strip("'")
            if s_crisp.get(feat) != val:
                return False
        elif ' IN ' in part:
            feat, val_str = part.split(' IN ')
            feat = feat.strip()
            try:
                val_list = ast.literal_eval(val_str.strip())
                if s_crisp.get(feat) not in val_list:
                    return False
            except Exception:
                return False
    return True

# 2. Hàm kiểm tra luật Continuous
def match_cont_rule(rule_text, s_cont):
    if rule_text == 'TRUE': return True
    parts = rule_text.split(' AND ')
    for part in parts:
        if ' <= ' in part:
            feat, val = part.split(' <= ')
            feat = feat.strip()
            val = float(val.strip())
            if s_cont.get(feat, 0.0) > val:
                return False
        elif ' > ' in part:
            feat, val = part.split(' > ')
            feat = feat.strip()
            val = float(val.strip())
            if s_cont.get(feat, 0.0) <= val:
                return False
    return True

# 3. Hàm xếp hạng Top 5 luật phù hợp nhất
def rank_top_rules(matched_rules, true_label, top_k=5):
    # Phân nhóm: luật dự đoán ĐÚNG (True Positive) và luật dự đoán SAI (False Positive/Cảnh báo)
    correct_rules   = []
    warning_rules   = []
    
    for r in matched_rules:
        is_correct = (r['consequent'] == true_label)
        rule_item = dict(r)
        rule_item['is_correct'] = is_correct
        rule_item['status'] = 'HỢP LỆ (Dự đoán đúng)' if is_correct else 'CẢNH BÁO (Dự đoán nhầm)'
        if is_correct:
            correct_rules.append(rule_item)
        else:
            warning_rules.append(rule_item)
            
    # Sắp xếp trong mỗi nhóm theo điểm số (score hoặc confidence)
    correct_rules.sort(key=lambda x: (-x.get('score', 0), -x.get('confidence', 0)))
    warning_rules.sort(key=lambda x: (-x.get('score', 0), -x.get('confidence', 0)))
    
    # Ưu tiên các luật đúng trước, sau đó bổ sung cảnh báo nếu có
    selected = correct_rules[:top_k]
    if len(selected) < top_k:
        selected.extend(warning_rules[:top_k - len(selected)])
        
    return selected

print('Bộ máy đối chiếu và xếp hạng luật đã sẵn sàng!')


## 4. Báo Cáo Chi Tiết Top 5 Luật Phù Hợp Cho Từng Mẫu (18 Mẫu Hold-out)


In [ ]:
explanation_results = []
summary_table_rows  = []

for idx in range(len(df_holdout_crisp)):
    s_crisp = df_holdout_crisp.iloc[idx].to_dict()
    s_cont  = df_holdout_continuous.iloc[idx].to_dict()
    
    true_label = s_crisp['label']
    audio_path = s_crisp['audio_path']
    
    # 1. Đối chiếu luật Crisp
    matched_crisp = [r for r in rf_crisp_rules if match_crisp_rule(r['rule_text'], s_crisp)]
    top5_crisp = rank_top_rules(matched_crisp, true_label, top_k=5)
    
    # 2. Đối chiếu luật Continuous
    matched_cont = [r for r in rf_cont_rules if match_cont_rule(r['rule_text'], s_cont)]
    top5_cont = rank_top_rules(matched_cont, true_label, top_k=5)
    
    # Lưu vào danh sách báo cáo chi tiết
    sample_res = {
        'sample_idx': idx,
        'label': true_label,
        'audio_path': audio_path,
        'crisp_profile': {k: s_crisp[k] for k in ['Tempo', 'Pitch', 'Energy', 'Vibrato', 'Timbre', 'Harmony', 'Spectral']},
        'top5_crisp_rules': top5_crisp,
        'top5_continuous_rules': top5_cont,
        'stats': {
            'total_matched_crisp': len(matched_crisp),
            'correct_matched_crisp': sum(1 for r in matched_crisp if r['consequent'] == true_label),
            'total_matched_cont': len(matched_cont),
            'correct_matched_cont': sum(1 for r in matched_cont if r['consequent'] == true_label),
        }
    }
    explanation_results.append(sample_res)
    
    # Bổ sung vào bảng phẳng CSV
    for rank_i, r in enumerate(top5_crisp, 1):
        summary_table_rows.append({
            'sample_idx': idx, 'true_label': true_label, 'audio_path': audio_path,
            'rule_type': 'Crisp', 'rank': rank_i, 'is_correct': r['is_correct'],
            'predicted_genre': r['consequent'], 'rule_text': r['rule_text'],
            'confidence': f"{r['confidence']*100:.1f}%", 'support': r.get('total_support', r.get('support', 0)),
            'tree_frequency': r.get('tree_frequency', 'N/A'), 'status': r['status']
        })
    for rank_i, r in enumerate(top5_cont, 1):
        summary_table_rows.append({
            'sample_idx': idx, 'true_label': true_label, 'audio_path': audio_path,
            'rule_type': 'Continuous', 'rank': rank_i, 'is_correct': r['is_correct'],
            'predicted_genre': r['consequent'], 'rule_text': r['rule_text'],
            'confidence': f"{r['confidence']*100:.1f}%", 'support': r.get('total_support', r.get('support', 0)),
            'tree_frequency': r.get('tree_frequency', 'N/A'), 'status': r['status']
        })

# Xuất kết quả
with open('./data/rules/holdout_top5_rules_explanation.json', 'w', encoding='utf-8') as f:
    json.dump(explanation_results, f, ensure_ascii=False, indent=2)

df_summary = pd.DataFrame(summary_table_rows)
df_summary.to_csv('./data/rules/holdout_top5_rules_summary.csv', index=False, encoding='utf-8-sig')

print('='*90)
print('ĐÃ HOÀN TẤT SUY LUẬN & GIẢI THÍCH CHO 18 MẪU HOLDOUT:')
print('  - ./data/rules/holdout_top5_rules_explanation.json (Chi tiết đầy đủ)')
print('  - ./data/rules/holdout_top5_rules_summary.csv (Bảng tổng hợp)')
print('='*90)

# In mẫu kết quả của 3 bài đại diện
print('\n' + '='*90)
print('TRÍCH ĐOẠN MINH HỌA GIẢI THÍCH SUY LUẬN CHO 2 MẪU TIÊU BIỂU:')
print('='*90)

for s in explanation_results[::6]: # Lấy 1 mẫu Ca trù và 1 mẫu Chầu văn
    print(f'\n[MẪU {s["sample_idx"]}] — Thể loại thực tế: {s["label"]} | File: {s["audio_path"]}')
    print('  Hồ sơ mờ (Crisp):', s['crisp_profile'])
    print(f'  Tổng số luật kích hoạt: {s["stats"]["total_matched_crisp"]} luật Crisp ({s["stats"]["correct_matched_crisp"]} đúng) | {s["stats"]["total_matched_cont"]} luật Continuous ({s["stats"]["correct_matched_cont"]} đúng)')
    print('  --- TOP 3 LUẬT CRISP PHÙ HỢP NHẤT ---')
    for r in s['top5_crisp_rules'][:3]:
        print(f'    * [{r["status"]}] IF {r["rule_text"]}')
        print(f'      -> DỰ ĐOÁN: {r["consequent"]} | Độ tin cậy: {r["confidence"]*100:.1f}% | Số mẫu: {r.get("total_support", 0)}')
    print('  --- TOP 2 LUẬT CONTINUOUS PHÙ HỢP NHẤT ---')
    for r in s['top5_continuous_rules'][:2]:
        print(f'    * [{r["status"]}] IF {r["rule_text"]}')
        print(f'      -> DỰ ĐOÁN: {r["consequent"]} | Độ tin cậy: {r["confidence"]*100:.1f}% | Số mẫu: {r.get("support", 0)}')


## 5. Trực Quan Hóa Tỷ Lệ Kích Hoạt Luật & Độ Khớp Của 18 Mẫu


In [ ]:
# Vẽ biểu đồ thống kê mức độ kích hoạt luật cho 18 mẫu hold-out
labels_sample = [f"Mẫu {s['sample_idx']}\n({s['label']})" for s in explanation_results]
crisp_correct = [s['stats']['correct_matched_crisp'] for s in explanation_results]
crisp_total   = [s['stats']['total_matched_crisp'] for s in explanation_results]
cont_correct  = [s['stats']['correct_matched_cont'] for s in explanation_results]
cont_total    = [s['stats']['total_matched_cont'] for s in explanation_results]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), facecolor='#0d1117')
fig.suptitle('Thống Kê Số Lượng Luật Mờ Kích Hoạt Trên 18 Mẫu Hold-out (Đúng vs Tổng số)', fontsize=15, fontweight='bold', color='#e6edf3')

x = np.arange(len(labels_sample))
w = 0.38

# Đồ thị Crisp
ax1.bar(x - w/2, crisp_total, w, label='Tổng số luật Crisp khớp', color='#45B7D1', alpha=0.5, edgecolor='#30363d')
ax1.bar(x - w/2, crisp_correct, w, label='Luật Crisp dự đoán ĐÚNG', color='#4ECDC4', edgecolor='#30363d')
ax1.set_title('Bộ Luật Random Forest Crisp (Ngôn Ngữ Tự Nhiên)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Số lượng luật')
ax1.set_xticks(x)
ax1.set_xticklabels(labels_sample, rotation=35, fontsize=8)
ax1.legend(loc='upper right', framealpha=0.6)
ax1.grid(alpha=0.2, linestyle='--')

# Đồ thị Continuous
ax2.bar(x + w/2, cont_total, w, label='Tổng số luật Continuous khớp', color='#FFEAA7', alpha=0.5, edgecolor='#30363d')
ax2.bar(x + w/2, cont_correct, w, label='Luật Continuous dự đoán ĐÚNG', color='#FF6B6B', edgecolor='#30363d')
ax2.set_title('Bộ Luật Random Forest Continuous (Ngưỡng Mờ Liên Tục)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Số lượng luật')
ax2.set_xticks(x)
ax2.set_xticklabels(labels_sample, rotation=35, fontsize=8)
ax2.legend(loc='upper right', framealpha=0.6)
ax2.grid(alpha=0.2, linestyle='--')

plt.tight_layout()
plt.savefig('./data/plots/13_holdout_rule_activation_summary.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Đã lưu biểu đồ: ./data/plots/13_holdout_rule_activation_summary.png')


## 6. Phân Tích Điểm Chung Cốt Lõi (Archetypes) & Tạo Bảng Tra Cứu Trực Quan

Từ Top các luật mờ được kích hoạt chính xác cho từng mẫu, ta tổng hợp được **đặc tính âm học đặc trưng chung (Archetype)** của từng thể loại âm nhạc truyền thống qua 7 trục ngôn ngữ mờ Crisp:
- **Ca trù**: Nhịp *Chậm - Vừa*, Âm vực *Trầm*, Năng lượng *Rất yếu - Yếu*, Âm sắc *Rất thấp*, Hòa âm *Thấp - Trung bình* (không gian thính phòng mộc mạc, tĩnh lặng).
- **Chèo**: Hòa âm *Cao* (dàn gõ và đàn nhị, sáo dày đặc), Âm vực *Trung - Cao*, Luyến láy *Vừa - Nhiều luyến* (giọng hát vang, nảy âm sân khấu truyền thống).
- **Chầu văn**: Âm vực *Rất trầm* (tiếng đàn Nguyệt gảy âm vực thấp dồn dập), Luyến láy *Ít luyến* (nhấn mạnh nhịp điệu dồn dập, thôi thúc hầu đồng), Hòa âm *Cao*.
- **Cải lương**: Luyến láy *Vừa - Nhiều luyến* (làn hơi vọng cổ ngân rung mùi mẫn), Âm sắc *Cao - Rất cao*, Hòa âm *Cao*, Nhịp *Chậm - Vừa*.
- **Hát xẩm**: Âm sắc *Rất thấp - Thấp*, Luyến láy *Ít luyến*, Phổ tần *Thấp - Vừa* (âm hưởng mộc mạc đường phố dân dã, lối hát tự sự kể chuyện).
- **Quan họ**: Năng lượng *Mạnh - Rất mạnh* (kỹ thuật Vang - Rền - Nền - Nảy), Nhịp *Vừa*, Âm sắc *Rất cao* (lối đối đáp giao duyên mộc mạc nhưng nội lực cột hơi rất khỏe).


In [ ]:
# Phân tích ma trận đặc tính cốt lõi (Archetype) và trực quan hóa Heatmap
genres = ['Ca trù', 'Chèo', 'Chầu văn', 'Cải lương', 'Hát xẩm', 'Quan họ']
axes = ['Tempo', 'Pitch', 'Energy', 'Vibrato', 'Timbre', 'Harmony', 'Spectral']

ORDINAL_SCALES = {
    'Tempo': {'Chậm': 0.2, 'Vừa': 0.5, 'Nhanh': 0.8},
    'Pitch': {'Rất trầm': 0.1, 'Trầm': 0.3, 'Trung': 0.5, 'Cao': 0.7, 'Rất cao': 0.9},
    'Energy': {'Rất yếu': 0.1, 'Yếu': 0.3, 'Vừa': 0.5, 'Mạnh': 0.7, 'Rất mạnh': 0.9},
    'Vibrato': {'Ít luyến': 0.2, 'Vừa': 0.5, 'Nhiều luyến': 0.8},
    'Timbre': {'Rất thấp': 0.1, 'Thấp': 0.3, 'Trung bình': 0.5, 'Cao': 0.7, 'Rất cao': 0.9},
    'Harmony': {'Thấp': 0.2, 'Trung bình': 0.5, 'Cao': 0.8},
    'Spectral': {'Thấp': 0.2, 'Trung bình': 0.5, 'Cao': 0.8},
}

genre_profiles_num = []
genre_profiles_text = []

for g in genres:
    samples = [s for s in explanation_results if s['label'] == g]
    row_num = []
    row_text = []
    for ax in axes:
        vals = [s['crisp_profile'][ax] for s in samples]
        mode_val = max(set(vals), key=vals.count)
        num_val = np.mean([ORDINAL_SCALES[ax].get(v, 0.5) for v in vals])
        row_num.append(num_val)
        row_text.append(f"{mode_val}")
    genre_profiles_num.append(row_num)
    genre_profiles_text.append(row_text)

df_num = pd.DataFrame(genre_profiles_num, index=genres, columns=axes)
df_text = pd.DataFrame(genre_profiles_text, index=genres, columns=axes)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [1.3, 1]})

sns.heatmap(df_num, annot=df_text, fmt='', cmap='magma', cbar=True,
            cbar_kws={'label': 'Mức độ định lượng mờ [0.0 - 1.0]'},
            linewidths=1.5, linecolor='#21262d', ax=ax1)
ax1.set_title('BẢNG ĐẶC TÍNH CỐT LÕI (ARCHETYPE) CỦA 6 THỂ LOẠI TỪ TOP LUẬT', fontsize=14, fontweight='bold', color='#58a6ff', pad=15)
ax1.set_ylabel('Thể loại', fontsize=12, fontweight='bold')
ax1.set_xlabel('Trục đặc trưng ngữ nghĩa', fontsize=12, fontweight='bold')

conf_by_genre = []
for g in genres:
    samples = [s for s in explanation_results if s['label'] == g]
    confs = [r['confidence'] for s in samples for r in s.get('top5_crisp_rules', []) if r['is_correct']]
    conf_by_genre.append(np.mean(confs) if confs else 0)

colors = ['#58a6ff', '#3fb950', '#d29922', '#f85149', '#bc8cff', '#39c5bb']
bars = ax2.barh(genres, conf_by_genre, color=colors, edgecolor='#ffffff', alpha=0.85, height=0.55)
ax2.set_xlim(0, 1.0)
ax2.set_xlabel('Độ tin cậy trung bình (Confidence) của Top luật đúng', fontsize=12, fontweight='bold')
ax2.set_title('ĐỘ TIN CẬY TRUNG BÌNH CỦA TOP LUẬT ĐƯỢC KÍCH HOẠT', fontsize=14, fontweight='bold', color='#58a6ff', pad=15)
for bar in bars:
    w = bar.get_width()
    ax2.text(w + 0.02, bar.get_y() + bar.get_height()/2, f'{w:.1%}', va='center', ha='left', fontsize=11, fontweight='bold', color='#e6edf3')

plt.tight_layout()
plt.savefig('./data/plots/14_genre_fuzzy_archetypes_common_patterns.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Đã lưu biểu đồ: ./data/plots/14_genre_fuzzy_archetypes_common_patterns.png')
print('-> Đã tạo Dashboard tra cứu HTML trực quan tại: ./data/plots/holdout_rules_dashboard.html')

# Tính toán và xuất bảng tần số điều kiện xuất hiện >= 3 luật
from collections import Counter
cond_rows = []
for g in genres:
    samples = [s for s in explanation_results if s['label'] == g]
    all_rules = [r for s in samples for r in s.get('top5_crisp_rules', [])]
    total_rules = len(all_rules)
    counter = Counter()
    for r in all_rules:
        for c in r['rule_text'].split(' AND '):
            counter[c.strip()] += 1
    for c, cnt in counter.most_common():
        if cnt >= 3:
            axis = c.split()[0] if ' ' in c else 'Khác'
            pct = round((cnt / total_rules) * 100, 1)
            cond_rows.append({
                'genre': g,
                'axis': axis,
                'condition': c,
                'frequency': cnt,
                'total_rules': total_rules,
                'percentage': f'{pct}%'
            })

df_cond_freq = pd.DataFrame(cond_rows)
df_cond_freq.to_csv('./data/rules/common_rule_conditions_frequency.csv', index=False, encoding='utf-8-sig')
print(f'-> Đã lưu bảng tần số điều kiện cốt lõi: ./data/rules/common_rule_conditions_frequency.csv ({len(df_cond_freq)} điều kiện có tần số >= 3)')
